In [14]:
from pathlib import Path

from ovito.io import import_file, export_file
from ovito.modifiers import CreateBondsModifier, CoordinationAnalysisModifier, ExpressionSelectionModifier, InvertSelectionModifier, DeleteSelectedModifier
import numpy as np

In [15]:
file_path = '/Users/loganyamamoto/Desktop/Research/grants/geo_sciences/finalized-bubble-collapse/structures/water/0013_19x1x1.data'
_path = Path(file_path)
shifted_path = _path.parent / f"{_path.stem}_shifted{_path.suffix}"
pipeline = import_file(file_path)
temp = 300
surface_dist = 1.2  # Å from top/bottom surfaces
bond_cutoff = 1.1
refit_box = True  # Shift atoms so min position is 0 and resize box to atom extent
refit_box_pad = 0.01  # Å of padding added to each face of the non-periodic axis after refit
                      # (prevents atoms from sitting exactly on xlo/xhi, which LAMMPS rejects)

# Non-periodic direction: 'x', 'y', or 'z' (slab normal / confined axis)
direction = 'x'
axis = {'x': 0, 'y': 1, 'z': 2}[direction]

# If True, atoms that are already bondless under full 3-D PBC are detected
# before surface-shifting and excluded from the shift candidates entirely.
# They remain in the exported structure at their original positions.
exclude_preexisting_orphans = True

In [16]:
# Ensure correct PBC: non-periodic along chosen direction, periodic in the other two
def set_pbc(frame, data):
    cell = data.cell_  # Use mutable version with underscore
    pbc = [True, True, True]
    pbc[axis] = False
    cell.pbc = tuple(pbc)

pipeline.modifiers.append(set_pbc)

In [17]:
# Create the modifier with Pairwise mode
bond_modifier = CreateBondsModifier(
    mode = CreateBondsModifier.Mode.Pairwise
)

# Set pairwise cutoffs using the set_pairwise_cutoff method
# Note: (1, 2) and (2, 1) are the same pair, so only need to set once
bond_modifier.set_pairwise_cutoff(1, 2, bond_cutoff)

pipeline.modifiers.append(bond_modifier)

In [18]:
# Coordination analysis based on existing bonds
# This modifier counts the number of bonds for each particle
def compute_coordination_from_bonds(frame, data):
    """Compute coordination number from existing bonds."""
    # Get bonds
    bonds = data.particles.bonds
    if bonds is None:
        raise RuntimeError("No bonds found. Make sure CreateBondsModifier runs before this.")
    
    # Get bond topology array (each row contains [particle1_index, particle2_index])
    topology = bonds.topology.array
    
    # Initialize coordination array
    num_particles = data.particles.count
    coordination = np.zeros(num_particles, dtype=int)
    
    # Count bonds for each particle
    # topology[:, 0] contains first particle indices, topology[:, 1] contains second particle indices
    np.add.at(coordination, topology[:, 0], 1)  # Increment coordination for first particles
    np.add.at(coordination, topology[:, 1], 1)  # Increment coordination for second particles
    
    # Store as particle property
    data.particles_.create_property('Coordination', data=coordination)

pipeline.modifiers.append(compute_coordination_from_bonds)

In [19]:
# Function to export selected subset (adds modifiers, exports, then removes them)
def export_subset(mask, filename):
    mask_copy = mask.copy()
    def select_subset(frame, data):
        data.particles_.create_property("Selection", data=mask_copy.astype(int))
    pipeline.modifiers.append(select_subset)
    pipeline.modifiers.append(InvertSelectionModifier())
    pipeline.modifiers.append(DeleteSelectedModifier())
    try:
        export_file(pipeline, filename, "lammps/dump",
                    columns = ["Particle Identifier", "Particle Type", "Position.X", "Position.Y", "Position.Z"])
    finally:
        for _ in range(3):
            pipeline.modifiers.pop()

# ---- Pre-existing orphan detection (full PBC) ----
# Run a separate pipeline with all axes periodic to find atoms that are
# bondless regardless of shifting.  These are never candidates for surface
# shifting — they stay at their original positions.
if exclude_preexisting_orphans:
    _pbc_pip = import_file(file_path)
    def _set_full_pbc(frame, data):
        data.cell_.pbc = (True, True, True)
    _bond_pbc = CreateBondsModifier(mode=CreateBondsModifier.Mode.Pairwise)
    _bond_pbc.set_pairwise_cutoff(1, 2, bond_cutoff)
    _pbc_pip.modifiers.append(_set_full_pbc)
    _pbc_pip.modifiers.append(_bond_pbc)
    _pbc_pip.modifiers.append(compute_coordination_from_bonds)
    _d_pbc = _pbc_pip.compute()
    _coord_pbc = _d_pbc.particles['Coordination'].array
    preexisting_orphan_ids = set(
        _d_pbc.particles['Particle Identifier'].array[_coord_pbc == 0].tolist()
    )
    print(f"Pre-existing orphans (full PBC, excluded from shifting): {len(preexisting_orphan_ids)}")
else:
    preexisting_orphan_ids = set()
    print("Pre-existing orphan exclusion disabled.")

# ---- Pass 1: compute bonds and coordination (no shifts yet) ----
data = pipeline.compute()

positions = data.particles.positions
coord = data.particles['Coordination'].array
axis_coords = positions[:, axis]
particle_ids = data.particles['Particle Identifier'].array
# Ovito cell: columns 0,1,2 = cell vectors a,b,c; column 3 = origin. Bounds = origin to origin + diagonal.
axis_min = data.cell[axis, 3]
axis_max = data.cell[axis, 3] + data.cell[axis, axis]
print(f"{direction}_min, {direction}_max:", axis_min, axis_max)
axis_mid = 0.5 * (axis_max + axis_min)

axis_box_size = axis_max - axis_min

# Exclude pre-existing orphans from the candidate pool so they are never shifted
preexisting_orphan_mask = np.isin(particle_ids, list(preexisting_orphan_ids))
free_mask = (coord == 0) & ~preexisting_orphan_mask

# Bottom surface: free atoms in bottom zone (select and move these first)
bottom_mask = free_mask & (axis_coords <= axis_mid) & (axis_coords <= axis_min + surface_dist)

# Export bottom free atoms and their IDs (from current state, before any shift)
export_subset(bottom_mask, "free_atoms_bottom.dump")
bottom_ids = particle_ids[bottom_mask]
with open("free_atoms_bottom_ids.txt", "w") as f:
    f.write(" || ".join(f"ParticleIdentifier == {pid}" for pid in bottom_ids))
print("Pass 1: saved bottom free atoms (before shifting).")

# ---- Shift bottom free atoms by +axis_box (so bonds are recomputed without them) ----
bottom_mask_copy = bottom_mask.copy()
def shift_bottom_along_axis(frame, data):
    pos = np.copy(data.particles.positions)
    pos[bottom_mask_copy, axis] += axis_box_size
    data.particles_.positions_ = pos
pipeline.modifiers.append(shift_bottom_along_axis)

# ---- Pass 2: recompute bonds and coordination after moving bottom ----
bond_modifier_2 = CreateBondsModifier(mode=CreateBondsModifier.Mode.Pairwise)
bond_modifier_2.set_pairwise_cutoff(1, 2, 1.2)
pipeline.modifiers.append(bond_modifier_2)
pipeline.modifiers.append(compute_coordination_from_bonds)

data2 = pipeline.compute()

# ---- Top surface: free atoms in top zone (re-checked after bottom move) ----
coord2 = data2.particles['Coordination'].array
positions2 = data2.particles.positions
axis_coords2 = positions2[:, axis]
free_mask2 = (coord2 == 0)
top_mask = free_mask2 & (axis_coords2 > axis_mid) & (axis_coords2 >= axis_max - surface_dist) & ~bottom_mask_copy

# Export top free atoms and their IDs (after bottom was moved)
export_subset(top_mask, "free_atoms_top.dump")
top_ids = data2.particles['Particle Identifier'].array[top_mask]
with open("free_atoms_top_ids.txt", "w") as f:
    f.write(" || ".join(f"ParticleIdentifier == {pid}" for pid in top_ids))
print("Pass 2: re-checked top surface and saved top free atoms.")
print("Saved particle ID filter strings to free_atoms_top_ids.txt and free_atoms_bottom_ids.txt")



Pre-existing orphans (full PBC, excluded from shifting): 798
x_min, x_max: -253.8522 282.058
Pass 1: saved bottom free atoms (before shifting).
Pass 2: re-checked top surface and saved top free atoms.
Saved particle ID filter strings to free_atoms_top_ids.txt and free_atoms_bottom_ids.txt


In [20]:
# Shift top free atoms by -axis_box and export full structure (bottom already shifted in cell 5)
top_mask_copy = top_mask.copy()

def shift_top_along_axis(frame, data):
    pos = np.copy(data.particles.positions)
    pos[top_mask_copy, axis] -= axis_box_size
    data.particles_.positions_ = pos

def shift_and_resize_box(frame, data):
    pos = np.copy(data.particles.positions)
    axis_min_val = pos[:, axis].min()
    axis_range = pos[:, axis].max() - axis_min_val

    # 1. Shift atoms: apply padding below so atoms start at refit_box_pad.
    # 2. Set box origin to 0 so xlo == 0 (recenter as final step).
    # LAMMPS rejects atoms sitting exactly on xlo/xhi during domain decomposition.
    pos[:, axis] -= axis_min_val - refit_box_pad
    data.particles_.positions_ = pos

    cell = data.cell_
    cell[axis, 3] = 0.0
    cell[axis, axis] = axis_range + 2 * refit_box_pad

pipeline.modifiers.append(shift_top_along_axis)
n_extra = 1
if refit_box:
    pipeline.modifiers.append(shift_and_resize_box)
    n_extra += 1
export_file(pipeline, str(shifted_path), "lammps/data")
for _ in range(n_extra):
    pipeline.modifiers.pop()
print(f"Saved modified structure to {shifted_path}")

Saved modified structure to /Users/loganyamamoto/Desktop/Research/grants/geo_sciences/finalized-bubble-collapse/structures/water/0013_19x1x1_shifted.data


In [21]:
# Verification: counts and pipeline state (run after cells 5 and 6)
assert bottom_mask.sum() == len(bottom_ids), "Bottom mask count vs saved IDs"
assert top_mask.sum() == len(top_ids), "Top mask count vs saved IDs"
print(f"Bottom free atoms: {bottom_mask.sum()} (IDs saved: {len(bottom_ids)})")
print(f"Top free atoms (after re-check): {top_mask.sum()} (IDs saved: {len(top_ids)})")
print(f"Pipeline modifiers after cell 6: {len(pipeline.modifiers)} (expected 6: pbc, bonds, coord, shift_bottom_along_axis, bonds2, coord2)")

Bottom free atoms: 33 (IDs saved: 33)
Top free atoms (after re-check): 28 (IDs saved: 28)
Pipeline modifiers after cell 6: 6 (expected 6: pbc, bonds, coord, shift_bottom_along_axis, bonds2, coord2)


In [22]:
# Load the exported shifted structure and verify positions and box bounds
shifted_pipeline = import_file(str(shifted_path))
data_shifted = shifted_pipeline.compute()
axis_all = data_shifted.particles.positions[:, axis]
box_origin = data_shifted.cell[axis, 3]
box_size = data_shifted.cell[axis, axis]
print(f"Exported file: {shifted_path!s}")
print(f"Minimum {direction} (all particles): {axis_all.min()}")
print(f"Maximum {direction} (all particles): {axis_all.max()}")
print(f"Box {direction} origin: {box_origin}, box {direction} size: {box_size}")

Exported file: /Users/loganyamamoto/Desktop/Research/grants/geo_sciences/finalized-bubble-collapse/structures/water/0013_19x1x1_shifted.data
Minimum x (all particles): 0.0
Maximum x (all particles): 537.4648550423
Box x origin: 0.0, box x size: 537.4648550423


In [23]:
axis_length = axis_all.max() - axis_all.min()
print(f"Length of {direction} axis: {axis_length}")

Length of x axis: 537.4648550423


In [24]:
# Validation: after all shifts/refit, every atom in the box has >= 1 bond.
# Pre-existing orphans (bondless even with full PBC on the input) are warned
# about but do NOT fail the check — the shifting logic cannot fix those.
# Any NEW orphans introduced by shifting DO fail.

shifted_pipeline = import_file(str(shifted_path))
shifted_pipeline.modifiers.append(set_pbc)
bond_modifier_final = CreateBondsModifier(mode=CreateBondsModifier.Mode.Pairwise)
bond_modifier_final.set_pairwise_cutoff(1, 2, bond_cutoff)
shifted_pipeline.modifiers.append(bond_modifier_final)
shifted_pipeline.modifiers.append(compute_coordination_from_bonds)

data_final = shifted_pipeline.compute()
coord_final = data_final.particles['Coordination'].array
final_orphan_ids = set(
    data_final.particles['Particle Identifier'].array[coord_final == 0].tolist()
)

# Determine which final orphans also existed in the original with full PBC.
# preexisting_orphan_ids is populated in cell 5 (requires exclude_preexisting_orphans=True
# or cell 11 having been run first to define orig_orphan_ids as a fallback).
_known_orphans = preexisting_orphan_ids if preexisting_orphan_ids else globals().get('orig_orphan_ids', set())
new_orphan_ids = final_orphan_ids - _known_orphans

n_final = len(final_orphan_ids)
n_preexisting = len(final_orphan_ids & _known_orphans)
n_new = len(new_orphan_ids)

print(f"Orphan atoms after shifts (coord==0): {n_final}")
print(f"  Pre-existing in original (full PBC): {n_preexisting}  [expected — input data issue, not a shifting bug]")
print(f"  Newly introduced by shifting        : {n_new}  [should be 0]")

if n_new:
    print("Newly-introduced orphan IDs (first 20):", sorted(new_orphan_ids)[:20])
    new_pos = {int(pid): pos.tolist()
               for pid, pos in zip(
                   data_final.particles['Particle Identifier'].array,
                   data_final.particles.positions)
               if int(pid) in new_orphan_ids}
    print("Their positions:")
    for pid, p in list(new_pos.items())[:20]:
        print(f"  ID {pid}  pos={p}")

assert n_new == 0, f"Shifting introduced {n_new} new orphan atoms. See IDs/positions above."

Orphan atoms after shifts (coord==0): 798
  Pre-existing in original (full PBC): 798  [expected — input data issue, not a shifting bug]
  Newly introduced by shifting        : 0  [should be 0]


In [25]:
# ── Diagnostic 1: were the orphans already bondless with full PBC? ──────────
# Reload the ORIGINAL file, keep x periodic as well, recompute bonds.
# Any atom with coord==0 here was orphaned before any shifting occurred.

orig_pipeline_pbc = import_file(file_path)

def set_full_pbc(frame, data):
    data.cell_.pbc = (True, True, True)

bond_mod_pbc = CreateBondsModifier(mode=CreateBondsModifier.Mode.Pairwise)
bond_mod_pbc.set_pairwise_cutoff(1, 2, bond_cutoff)

orig_pipeline_pbc.modifiers.append(set_full_pbc)
orig_pipeline_pbc.modifiers.append(bond_mod_pbc)
orig_pipeline_pbc.modifiers.append(compute_coordination_from_bonds)

data_orig_pbc = orig_pipeline_pbc.compute()
coord_orig_pbc = data_orig_pbc.particles['Coordination'].array
orig_orphan_mask = (coord_orig_pbc == 0)
orig_orphan_ids = set(data_orig_pbc.particles['Particle Identifier'].array[orig_orphan_mask].tolist())

print(f"Orphans in ORIGINAL structure (full PBC, cutoff={bond_cutoff} Å): {len(orig_orphan_ids)}")

# Cross-reference against the final-shifted orphan IDs
data_final = shifted_pipeline.compute()   # shifted_pipeline built in cell 10
coord_final = data_final.particles['Coordination'].array
final_orphan_ids = set(
    data_final.particles['Particle Identifier'].array[coord_final == 0].tolist()
)
already_orphaned = orig_orphan_ids & final_orphan_ids
new_orphans = final_orphan_ids - orig_orphan_ids

print(f"Final orphans that were ALREADY orphaned with full PBC : {len(already_orphaned)}")
print(f"New orphans introduced by shifting (should be 0)       : {len(new_orphans)}")
if new_orphans:
    print("  Newly-introduced orphan IDs (first 20):", sorted(new_orphans)[:20])

Orphans in ORIGINAL structure (full PBC, cutoff=1.1 Å): 798
Final orphans that were ALREADY orphaned with full PBC : 798
New orphans introduced by shifting (should be 0)       : 0


In [26]:
# ── Diagnostic 2: double-shift check ─────────────────────────────────────────
# After pass 1, bottom-mask atoms are displaced to x + axis_box_size.
# In pass 2 they have coord=0 (outside box) and x ≥ axis_max - surface_dist,
# so they can slip into top_mask and get shifted a second time.
#
# We check: is bottom_mask ∩ top_mask non-empty?

double_shifted_mask = bottom_mask & top_mask   # both defined in cell 5
n_double = int(double_shifted_mask.sum())
print(f"Atoms in BOTH bottom_mask AND top_mask (shifted twice): {n_double}")

if n_double:
    ds_ids = particle_ids[double_shifted_mask]
    ds_orig_pos = positions[double_shifted_mask]  # original positions from Pass-1 data
    print("First 20 double-shifted Particle Identifiers:", ds_ids[:20].tolist())
    print("Their original positions (pre-shift):")
    for pid, p in zip(ds_ids[:20], ds_orig_pos[:20]):
        print(f"  ID {int(pid):>10}  {direction}={p[axis]:.4f}  (box goes {axis_min:.2f}→{axis_max:.2f})")
else:
    print("No double-shifted atoms — each atom is moved at most once.")

Atoms in BOTH bottom_mask AND top_mask (shifted twice): 0
No double-shifted atoms — each atom is moved at most once.
